In [1]:
import time
import json

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver import ActionChains
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

from webdriver_manager.chrome import ChromeDriverManager

from pydantic import BaseModel, ConfigDict
from typing import Optional
from pydantic import BaseModel, Field
from pydantic import ValidationError

In [2]:
class BPSScraper:
    def __init__(self):
        """Initialize the BPS scraper with Selenium WebDriver"""
        self.driver = None
        self.base_url = "http://discoverbps.bostonpublicschools.org"
        
    def start(self):
        """Start the WebDriver"""
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service)
        self.driver.get(self.base_url)
        
    def close(self):
        """Close the WebDriver"""
        if self.driver:
            self.driver.quit()

In [3]:
def keep_clicking(driver, button_id, target_element_id, click_interval=2, max_attempts=20):
    """
    Repeatedly clicks on a button (identified by button_id) until the target element
    (identified by target_element_id) appears, or until max_attempts is reached.

    If the button is not found on the page, the function prints a message and returns False.
    
    Args:
        driver (WebDriver): The Selenium WebDriver instance.
        button_id (str): The ID of the button that should be clicked.
        target_element_id (str): The ID of the target element to appear.
        click_interval (int, optional): Seconds to wait between clicks. Defaults to 2.
        max_attempts (int, optional): Maximum number of clicks before giving up. Defaults to 20.
    
    Returns:
        bool: True if the target element was found; otherwise, False.
    """
    attempts = 0
    while attempts < max_attempts:
        # First, check if the target element is present and visible.
        try:
            target = driver.find_element(By.ID, target_element_id)
            if target.is_displayed():
                print(f"Target element '{target_element_id}' found after {attempts} attempts.")
                return True
        except NoSuchElementException:
            pass

        # Try to click the button.
        try:
            time.sleep(1)
            button = driver.find_element(By.ID, button_id)
            button.click()
            print(f"Clicked button '{button_id}' (attempt {attempts + 1}).")
        except NoSuchElementException:
            print(f"Button '{button_id}' not found. It might not be available on this page.")
            return False  # Or decide to continue, depending on your use case.
        except Exception as e:
            print(f"Error clicking button '{button_id}' at attempt {attempts + 1}: {e}")

        time.sleep(click_interval)
        attempts += 1

    print(f"Target element '{target_element_id}' not found after {max_attempts} attempts.")
    return False

# Usage example:
# For a button with id "home_search_button" waiting for "addresses_next_button":
# if keep_clicking(driver, "home_search_button", "addresses_next_button", click_interval=2, max_attempts=20):
#     # Proceed with next steps, e.g., click "addresses_next_button"
#     driver.find_element(By.ID, "addresses_next_button").click()

In [4]:
# ORIGINAL FUNCTIONS
def scrape_school_details(school_details):
    container_elems = school_details.find_elements(By.CSS_SELECTOR, "p")

    # Create a list for pairing titles and descriptions.
    pairs = {}

    first_p = school_details.find_element(By.CSS_SELECTOR, "p")
    lines = first_p.text.splitlines()
    address = lines[0].strip() if lines else ""

    pairs['Address'] = address

    website_link = school_details.find_element(By.CSS_SELECTOR, 'a[title="School Website"]')
    pairs['School Website'] = website_link.get_attribute("href")

    email_link = first_p.find_element(By.CSS_SELECTOR, 'a[title="Email Address"]')
    email_href = email_link.get_attribute("href").split(':')[1]
    pairs['Email'] = email_href

    pairs['Distance from home'] = container_elems[1].text

    current_title = None

    for elem in container_elems[2:]:
        classes = elem.get_attribute("class").split()
        text = elem.text.strip()

        # If this is a title paragraph and has non-empty text, store it.
        if "title" in classes and text:
            current_title = text
        # If it's a description paragraph (with both 'descrip' and 'light')
        # and we have a stored title, pair them.
        elif "descrip" in classes and "light" in classes and current_title:
            pairs[current_title] = text
            current_title = None

    return pairs

def scrape_school_info(school_info):
    output_dict = {}
    output_dict['School Description'] = school_info.find_element(By.CLASS_NAME, 'school_description').text
    attribute_lst = school_info.find_elements(By.CLASS_NAME, 'box')

    for el in attribute_lst:
        key = el.find_element(By.CLASS_NAME, 'title').text
        value = el.find_element(By.CLASS_NAME, 'descrip').text
        output_dict[key] = value

    return output_dict

def scrape_school(school_el):

    output_dict = {}

    output_dict['School Name'] = school_el.find_element(By.CLASS_NAME, 'school_name').text
    
    school_details = school_el.find_element(By.CLASS_NAME, 'school_details')
    output_dict.update(scrape_school_details(school_details))
    
    school_info = school_el.find_element(By.CLASS_NAME, 'school_info')
    output_dict.update(scrape_school_info(school_info))

    return output_dict

In [5]:
# ACTUAL FUNCTIONS

def scrape_school_details(school_details):
    """
    Extracts school details information from the school_details container.
    
    Expected keys in the output dictionary:
      - Address: The first line of the first <p> element.
      - School Website: The href of the <a> with title "School Website".
      - Email: The email address from the <a> with title "Email Address" (without the mailto: prefix).
      - Distance from home: Taken from container's second <p> element.
      - Any additional title/description pairs (e.g. "School Hours", "Preview Dates", "Surround Care").
    """
    details = {}
    
    # Get all <p> elements within this container
    try:
        container_elems = school_details.find_elements(By.CSS_SELECTOR, "p")
    except Exception:
        container_elems = []
    
    # Extract the first paragraph for Address, Website, and Email.
    try:
        first_p = school_details.find_element(By.CSS_SELECTOR, "p")
        lines = first_p.text.splitlines()
        address = lines[0].strip() if lines and lines[0].strip() else ""
        details["Address"] = address
    except NoSuchElementException:
        details["Address"] = ""
    
    # Extract School Website from the first paragraph
    try:
        website_link = school_details.find_element(By.CSS_SELECTOR, 'a[title="School Website"]')
        website_href = website_link.get_attribute("href") or ""
        details["School Website"] = website_href
    except NoSuchElementException:
        details["School Website"] = ""
    
    # Extract Email from the first paragraph (remove the 'mailto:' prefix if present)
    try:
        email_link = first_p.find_element(By.CSS_SELECTOR, 'a[title="Email Address"]')
        email_href = email_link.get_attribute("href") or ""
        if email_href.startswith("mailto:"):
            email_href = email_href[len("mailto:"):]
        details["Email"] = email_href
    except NoSuchElementException:
        details["Email"] = ""
    
    # Extract Distance from home from the second <p> element, if available.
    if len(container_elems) > 1:
        details["Distance from home"] = container_elems[1].text.strip()
    else:
        details["Distance from home"] = ""
    
    # Now iterate through the remaining <p> elements (if any) for title/description pairs.
    current_title = None
    for elem in container_elems[2:]:
        try:
            classes = elem.get_attribute("class").split()
        except Exception:
            classes = []
        text = elem.text.strip() if elem.text else ""
        
        # If this is a title paragraph and has non-empty text, store it as the current title.
        if "title" in classes and text:
            current_title = text
        # If it is a description paragraph (both "descrip" and "light" in its classes) and we have a current title, save it.
        elif "descrip" in classes and "light" in classes and current_title:
            details[current_title] = text
            current_title = None
            
    return details


def scrape_school_info(school_info):
    """
    Extracts additional school information from the school_info container.
    
    Expected keys in the output dictionary include:
      - School Description
      - Any additional attributes grouped in boxes (e.g., "Grades Offered", "Demand Reports", etc.)
    
    If an element is missing, its corresponding value will be set to an empty string.
    """
    info = {}
    
    # Get School Description
    try:
        desc_elem = school_info.find_element(By.CLASS_NAME, 'school_description')
        info["School Description"] = desc_elem.text.strip()
    except NoSuchElementException:
        info["School Description"] = ""
    
    # Each attribute is contained in a box with a title and a description.
    try:
        attribute_lst = school_info.find_elements(By.CLASS_NAME, 'box')
        for el in attribute_lst:
            try:
                key = el.find_element(By.CLASS_NAME, 'title').text.strip()
            except NoSuchElementException:
                key = ""
            try:
                value = el.find_element(By.CLASS_NAME, 'descrip').text.strip()
            except NoSuchElementException:
                value = ""
            if key:
                info[key] = value
        # Ensure that optional keys such as "Community Partners" exist even if empty.
        if "Community Partners" not in info:
            info["Community Partners"] = ""
    except Exception:
        pass
    
    return info


def scrape_school(school_el):
    """
    Combines the information from school_details and school_info along with the School Name.
    
    Expected keys (per our schema) include:
      - School Name
      - Address
      - School Website
      - Email
      - Distance from home
      - School Description
      - And all additional attribute pairs from both school_details and school_info.
    
    For any missing element, an empty string is recorded.
    """
    school_data = {}
    
    # Extract School Name
    try:
        school_name_elem = school_el.find_element(By.CLASS_NAME, 'school_name')
        school_data["School Name"] = school_name_elem.text.strip()
    except NoSuchElementException:
        school_data["School Name"] = ""
    
    # Extract school_details section
    try:
        school_details = school_el.find_element(By.CLASS_NAME, 'school_details')
        details_dict = scrape_school_details(school_details)
        school_data.update(details_dict)
    except NoSuchElementException:
        # If missing, set default values for keys we expect from school_details.
        school_data.setdefault("Address", "")
        school_data.setdefault("School Website", "")
        school_data.setdefault("Email", "")
        school_data.setdefault("Distance from home", "")
    
    # Extract school_info section
    try:
        school_info = school_el.find_element(By.CLASS_NAME, 'school_info')
        info_dict = scrape_school_info(school_info)
        school_data.update(info_dict)
    except NoSuchElementException:
        # Ensure default for School Description in case it's missing.
        school_data.setdefault("School Description", "")
    
    return school_data

In [6]:
# PYDANTIC SCHEMA
class SchoolDetails(BaseModel):
    school_name: str = Field(..., alias="School Name")
    address: str = Field(..., alias="Address")
    school_website: Optional[str] = Field(None, alias="School Website")
    email: Optional[str] = Field(None, alias="Email")
    distance_from_home: str = Field(..., alias="Distance from home")
    school_hours: str = Field(..., alias="School Hours")
    preview_dates: str = Field(..., alias="Preview Dates")
    surround_care: str = Field(..., alias="Surround Care")
    school_description: str = Field(..., alias="School Description")
    grades_offered: str = Field(..., alias="Grades Offered")
    demand_reports: str = Field(..., alias="Demand Reports")
    eligibility: str = Field(..., alias="Eligibility")
    special_application: str = Field(..., alias="Special Application")
    quality: str = Field(..., alias="Quality")
    uniform_policy: str = Field(..., alias="Uniform Policy")
    school_focus: str = Field(..., alias="School Focus")
    programs: str = Field(..., alias="Programs")
    facility_features: str = Field(..., alias="Facility Features")
    student_support: str = Field(..., alias="Student Support")
    sports: str = Field(..., alias="Sports")
    community_partners: Optional[str] = Field(None, alias="Community Partners")

    class Config:
        # Allow population by alias names (i.e. keys from your scraped dictionary)
        allow_population_by_field_name = True

/Users/johnathansun/mambaforge/envs/mit_chatbot/lib/python3.10/site-packages/pydantic/_internal/_config.py:373: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'validate_by_name'
  warnings.warn(message, UserWarning)


In [169]:
student_data = {
    'grade': 8,
    'street_number' : '13',
    'street_name': 'thacher ct',
    'zip_code': '02113'
}

In [170]:
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# get driver to open link http://discoverbps.bostonpublicschools.org/
driver.get("http://discoverbps.bostonpublicschools.org/")

# Wait up to 20 seconds for the document to be fully loaded.
WebDriverWait(driver, 20).until(lambda d: d.execute_script("return document.readyState") == "complete")
print("Page is fully loaded.")

grade_el = driver.find_element(By.ID, "grade_level_placeholder")
street_num_el = driver.find_element(By.ID, "street_number")
street_el = driver.find_element(By.ID, "street_name")
zipcode_el = driver.find_element(By.ID, "zipcode")

time.sleep(0.5)

grade_el.send_keys(str(student_data['grade']))
street_num_el.send_keys(str(student_data['street_number']))
street_el.send_keys(str(student_data['street_name']))
zipcode_el.send_keys(str(student_data['zip_code']))

keep_clicking(driver, "home_search_button", "addresses_next_button")
time.sleep(2)
keep_clicking(driver, "addresses_next_button", "ell_next_button")
time.sleep(2)
keep_clicking(driver, "ell_next_button", "sped_next_button")
time.sleep(2)
keep_clicking(driver, "sped_next_button", "home_schools_list")

time.sleep(3)

home_schools_lst = driver.find_elements(By.CSS_SELECTOR, "ul#sortable > li.list_row.home_school.clearfix")

time.sleep(3)

school_data_lst = []

for el in home_schools_lst:
    el.find_element(By.CLASS_NAME, 'sortable_school_name').click()
    time.sleep(0.3)
    try:
        scraped_info = scrape_school(el)
        school = SchoolDetails.model_validate(scraped_info)
        print('Success for', school.school_name)
        school_data_lst.append(school)
    except ValidationError as e:
        print("Validation error:", e)

Page is fully loaded.
Clicked button 'home_search_button' (attempt 1).
Clicked button 'home_search_button' (attempt 2).
Clicked button 'home_search_button' (attempt 3).
Target element 'addresses_next_button' found after 3 attempts.
Clicked button 'addresses_next_button' (attempt 1).
Clicked button 'addresses_next_button' (attempt 2).
Target element 'ell_next_button' found after 2 attempts.
Clicked button 'ell_next_button' (attempt 1).
Target element 'sped_next_button' found after 1 attempts.
Clicked button 'sped_next_button' (attempt 1).
Target element 'home_schools_list' found after 1 attempts.
Success for Eliot K-8 Innovation School
Success for New Mission High School
Success for Warren-Prescott K-8 School
Success for Charlestown High School
Success for McKay K-8 School
Success for Mario Umana Academy
Success for Quincy Upper School
Success for East Boston High School
Success for Condon K-8 School
Success for Hurley K-8 School
Success for Dearborn STEM Academy
Success for Ruth Batson

In [171]:
school_data_lst

[SchoolDetails(school_name='Eliot K-8 Innovation School', address='16 Charter St Boston MA 02113', school_website='http://bostonpublicschools.org/Page/628', email='eliot@bostonpublicschools.org', distance_from_home='0.300 mi from home', school_hours='8:30am - 3:30pm', preview_dates='(P) In-Person Session; (V) - Virtual Session\n• 11/19/2024, 9:00 AM - 10:00 AM, (P)\n• 12/10/2024, 9:00 AM - 10:00 AM, (P)\n• 1/28/2025, 9:00 AM - 10:00 AM, (P)', surround_care='After: Champions After School program 3:30pm-6:00pm for the 2023-2024 school year for grades K-4', school_description='We provide an inclusive, joyful learning journey preparing students to achieve their highest potential by embracing their identities, developing interdisciplinary 21st century skills, empowered by knowledge to participate actively in a complex and constantly changing, culturally diverse world.', grades_offered='K0 - 8', demand_reports='See Seat and applicant data:\nLatest Demand Report\nHistoric Demand Data', eligib

In [2]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
from pydantic import BaseModel, Field, ValidationError

# Define the Pydantic schema.
class SchoolDetails(BaseModel):
    school_name: str = Field(..., alias="School Name")
    address: str = Field(..., alias="Address")
    school_website: str = Field("", alias="School Website")
    email: str = Field("", alias="Email")
    distance_from_home: str = Field(..., alias="Distance from home")
    school_hours: str = Field("", alias="School Hours")
    preview_dates: str = Field("", alias="Preview Dates")
    surround_care: str = Field("", alias="Surround Care")
    school_description: str = Field("", alias="School Description")
    grades_offered: str = Field("", alias="Grades Offered")
    demand_reports: str = Field("", alias="Demand Reports")
    eligibility: str = Field("", alias="Eligibility")
    special_application: str = Field("", alias="Special Application")
    quality: str = Field("", alias="Quality")
    uniform_policy: str = Field("", alias="Uniform Policy")
    school_focus: str = Field("", alias="School Focus")
    programs: str = Field("", alias="Programs")
    facility_features: str = Field("", alias="Facility Features")
    student_support: str = Field("", alias="Student Support")
    sports: str = Field("", alias="Sports")
    community_partners: str = Field("", alias="Community Partners")

    class Config:
        allow_population_by_field_name = True


class BPSScraperAgent:
    def __init__(self, student_data: dict):
        """
        student_data: a dictionary containing keys:
          'grade', 'street_number', 'street_name', 'zip_code'
        """
        self.student_data = student_data
        self.base_url = "http://discoverbps.bostonpublicschools.org"
        self.driver = None

    def start(self):
        """Starts the Selenium WebDriver and opens the target URL."""
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service)
        self.driver.get(self.base_url)
        # Wait until the page is fully loaded.
        WebDriverWait(self.driver, 20).until(
            lambda d: d.execute_script("return document.readyState") == "complete"
        )
        print("Page is fully loaded.")

    def fill_form(self):
        """Fills in the search form using the student_data provided."""
        grade_el = self.driver.find_element(By.ID, "grade_level_placeholder")
        street_num_el = self.driver.find_element(By.ID, "street_number")
        street_el = self.driver.find_element(By.ID, "street_name")
        zipcode_el = self.driver.find_element(By.ID, "zipcode")

        # A short pause to let elements settle.
        time.sleep(0.5)
        grade_el.send_keys(str(self.student_data['grade']))
        street_num_el.send_keys(str(self.student_data['street_number']))
        street_el.send_keys(str(self.student_data['street_name']))
        zipcode_el.send_keys(str(self.student_data['zip_code']))

    def keep_clicking(self, button_id, target_element_id, click_interval=2, max_attempts=20) -> bool:
        """
        Repeatedly clicks on a button (by button_id) until the target element (by target_element_id)
        appears, or until max_attempts is reached.
        """
        attempts = 0
        while attempts < max_attempts:
            try:
                target = self.driver.find_element(By.ID, target_element_id)
                if target.is_displayed():
                    print(f"Target element '{target_element_id}' found after {attempts} attempts.")
                    return True
            except NoSuchElementException:
                pass

            try:
                time.sleep(1)  # Short pause before clicking.
                button = self.driver.find_element(By.ID, button_id)
                button.click()
                print(f"Clicked button '{button_id}' (attempt {attempts + 1}).")
            except NoSuchElementException:
                print(f"Button '{button_id}' not found. It might not be available on this page.")
                return False
            except Exception as e:
                print(f"Error clicking button '{button_id}' at attempt {attempts + 1}: {e}")

            time.sleep(click_interval)
            attempts += 1

        print(f"Target element '{target_element_id}' not found after {max_attempts} attempts.")
        return False

    def scrape_school_details(self, school_details):
        """
        Extracts school details information from the school_details container.
        Expected keys include:
            - Address: First line of the first <p> element.
            - School Website: href of the <a> with title "School Website".
            - Email: Email address from the <a> with title "Email Address" (without the mailto:).
            - Distance from home: From the second <p>.
            - Plus any additional title/description pairs.
        """
        details = {}
        try:
            container_elems = school_details.find_elements(By.CSS_SELECTOR, "p")
        except Exception:
            container_elems = []
        try:
            first_p = school_details.find_element(By.CSS_SELECTOR, "p")
            lines = first_p.text.splitlines()
            address = lines[0].strip() if lines and lines[0].strip() else ""
            details["Address"] = address
        except NoSuchElementException:
            details["Address"] = ""
        try:
            website_link = school_details.find_element(By.CSS_SELECTOR, 'a[title="School Website"]')
            website_href = website_link.get_attribute("href") or ""
            details["School Website"] = website_href
        except NoSuchElementException:
            details["School Website"] = ""
        try:
            email_link = first_p.find_element(By.CSS_SELECTOR, 'a[title="Email Address"]')
            email_href = email_link.get_attribute("href") or ""
            if email_href.startswith("mailto:"):
                email_href = email_href[len("mailto:"):]
            details["Email"] = email_href
        except NoSuchElementException:
            details["Email"] = ""
        if len(container_elems) > 1:
            details["Distance from home"] = container_elems[1].text.strip()
        else:
            details["Distance from home"] = ""
        current_title = None
        for elem in container_elems[2:]:
            try:
                classes = elem.get_attribute("class").split()
            except Exception:
                classes = []
            text = elem.text.strip() if elem.text else ""
            if "title" in classes and text:
                current_title = text
            elif "descrip" in classes and "light" in classes and current_title:
                details[current_title] = text
                current_title = None
        return details

    def scrape_school_info(self, school_info):
        """
        Extracts additional school information from the school_info container.
        Expected keys include:
            - School Description
            - Any attributes found in boxes (e.g., Grades Offered, Demand Reports, etc.)
        """
        info = {}
        try:
            desc_elem = school_info.find_element(By.CLASS_NAME, 'school_description')
            info["School Description"] = desc_elem.text.strip()
        except NoSuchElementException:
            info["School Description"] = ""
        try:
            attribute_lst = school_info.find_elements(By.CLASS_NAME, 'box')
            for el in attribute_lst:
                try:
                    key = el.find_element(By.CLASS_NAME, 'title').text.strip()
                except NoSuchElementException:
                    key = ""
                try:
                    value = el.find_element(By.CLASS_NAME, 'descrip').text.strip()
                except NoSuchElementException:
                    value = ""
                if key:
                    info[key] = value
            if "Community Partners" not in info:
                info["Community Partners"] = ""
        except Exception:
            pass
        return info

    def scrape_school(self, school_el):
        """
        Combines information from school_details and school_info, along with the School Name.
        Returns a dictionary corresponding to one school's data.
        """
        school_data = {}
        try:
            school_name_elem = school_el.find_element(By.CLASS_NAME, 'school_name')
            school_data["School Name"] = school_name_elem.text.strip()
        except NoSuchElementException:
            school_data["School Name"] = ""
        try:
            school_details = school_el.find_element(By.CLASS_NAME, 'school_details')
            details_dict = self.scrape_school_details(school_details)
            school_data.update(details_dict)
        except NoSuchElementException:
            school_data.setdefault("Address", "")
            school_data.setdefault("School Website", "")
            school_data.setdefault("Email", "")
            school_data.setdefault("Distance from home", "")
        try:
            school_info = school_el.find_element(By.CLASS_NAME, 'school_info')
            info_dict = self.scrape_school_info(school_info)
            school_data.update(info_dict)
        except NoSuchElementException:
            school_data.setdefault("School Description", "")
        return school_data

    def scrape_schools(self):
        """
        Finds the list of school elements on the page,
        clicks each to open its details, and scrapes the data.
        Returns a list of validated SchoolDetails instances.
        """
        time.sleep(3)
        home_schools_lst = self.driver.find_elements(
            By.CSS_SELECTOR, "ul#sortable > li.list_row.home_school.clearfix"
        )
        time.sleep(3)
        school_data_lst = []
        for el in home_schools_lst:
            try:
                # Click the school name element to open details.
                el.find_element(By.CLASS_NAME, 'sortable_school_name').click()
                time.sleep(0.3)
                scraped_info = self.scrape_school(el)
                try:
                    # Validate using the Pydantic model.
                    school = SchoolDetails.model_validate(scraped_info)
                except ValidationError as e:
                    print("Validation error:", e)
                    continue
                print('Success for', school.school_name)
                school_data_lst.append(school)
            except Exception as e:
                print("Error processing a school element:", e)
        return school_data_lst

    def run(self):
        """
        Main method to run the scraper:
         - Starts the driver and loads the page.
         - Fills out the search form.
         - Navigates through multi-step process by repeatedly clicking buttons.
         - Scrapes and returns a list of school details.
        """
        self.start()
        # Ensure the page is fully loaded.
        WebDriverWait(self.driver, 20).until(
            lambda d: d.execute_script("return document.readyState") == "complete"
        )
        print("Page is fully loaded.")
        self.fill_form()
        # Navigate through the form steps.
        self.keep_clicking("home_search_button", "addresses_next_button")
        time.sleep(1)
        self.keep_clicking("addresses_next_button", "ell_next_button")
        time.sleep(1)
        self.keep_clicking("ell_next_button", "sped_next_button")
        time.sleep(1)
        self.keep_clicking("sped_next_button", "home_schools_list")
        time.sleep(1)
        # Scrape the school list.
        schools = self.scrape_schools()
        return schools

    def close(self):
        if self.driver:
            self.driver.quit()

/Users/johnathansun/mambaforge/envs/mit_chatbot/lib/python3.10/site-packages/pydantic/_internal/_config.py:373: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'validate_by_name'
  warnings.warn(message, UserWarning)


In [4]:
# ----- USAGE EXAMPLE -----
student_data = {
    'grade': 1,
    'street_number': '50',
    'street_name': 'everett st',
    'zip_code': '02128'
}

agent = BPSScraperAgent(student_data)
school_list = agent.run()
print("Scraped", len(school_list), "schools.")
agent.close()

Page is fully loaded.
Page is fully loaded.
Clicked button 'home_search_button' (attempt 1).
Target element 'addresses_next_button' found after 1 attempts.
Clicked button 'addresses_next_button' (attempt 1).
Clicked button 'addresses_next_button' (attempt 2).
Target element 'ell_next_button' found after 2 attempts.
Clicked button 'ell_next_button' (attempt 1).
Target element 'sped_next_button' found after 1 attempts.
Clicked button 'sped_next_button' (attempt 1).
Target element 'home_schools_list' found after 1 attempts.
Success for McKay K-8 School
Success for Adams Elementary School
Success for East Boston Early Education Center
Success for Alighieri Dante Montessori School
Success for Otis Elementary School
Success for Mario Umana Academy
Success for O'Donnell Elementary School
Success for Kennedy Patrick J Elementary School
Success for UP Academy Dorchester
Success for Eliot K-8 Innovation School
Success for Harvard-Kent Elementary School
Success for Guild Elementary School
Success